# 📋 미션 소개: Hugging Face를 활용한 문서 요약 모델 구현

## 1. 미션 개요
* **목표**: `Hugging Face transformers` 라이브러리를 활용하여 문서 요약 모델을 직접 구현합니다.
* **핵심 과제**: 데이터 로드 $\rightarrow$ 전처리 $\rightarrow$ 모델 학습(Fine-tuning) $\rightarrow$ 결과 평가로 이어지는 **전체 파이프라인(End-to-End Pipeline)**을 구축하는 것이 목표입니다.

## 2. 사용 데이터셋
* **출처**: AI HUB 문서요약 텍스트
* **형식**: JSON 파일 (약 0.4GB)
* **구성**:
    * **3가지 카테고리**: 신문 기사, 사설, 법률 문서
    * **구조**: 각 카테고리별 `Train`(학습) / `Test`(평가) 데이터 쌍으로 구성
    * **활용**: 전체 데이터를 사용하거나, 특정 카테고리를 선택하여 학습 가능

## 3. 단계별 가이드라인

### 1) 데이터 로드 및 전처리
* **데이터 파싱**: JSON 파일에서 본문(Text)과 요약문(Summary) 추출
* **클렌징**: 불필요한 기호, 공백 제거 등 텍스트 정제
* **형식 변환**: 모델 입력에 맞게 토큰화(Tokenization) 및 길이 조정

### 2) 모델 선택 및 실행
* **라이브러리**: Hugging Face `transformers` 활용
* **모델**: 사전 학습된(Pre-trained) 모델 사용 (예: T5, BART 등)
* **학습**: 주어진 데이터를 사용하여 모델을 미세 조정(**Fine-tuning**)

### 3) 모델 평가 및 결과 분석
* **정량적 평가**: **ROUGE** 등의 지표를 사용하여 요약 품질을 수치로 분석
* **정성적 평가**: 테스트 문장을 모델에 넣어 실제 생성된 요약문과 원본 비교

## 4. 제출 및 작성 필수 사항
* **제출 형식**: Google Colab Notebook (`.ipynb`)
* **파일 명**: `12_{팀명}_{성함}.ipynb`
* **마크다운(Markdown) 작성 필수**:
    * 단순 코드 제출이 아닌, 각 코드 셀의 **의도, 알고리즘, 함수 설명**을 마크다운 셀에 기록
    * 전체 워크플로우를 제3자가 보고 이해할 수 있도록 체계적으로 정리

## 5. 참고 사항
* **Baseline 코드**: 기본 제공 코드는 참고용일 뿐입니다. 이를 그대로 쓰기보다 **자신의 아이디어를 더해 발전**시키는 것이 중요합니다.
* **도전 과제**: 다양한 모델을 시도하거나 전처리 방식을 변경하여 성능을 높여보세요.

# 1. 환경 설정 및 데이터 로드 함수 정의

## 1) 필수 라이브러리 및 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install datasets
!pip install konlpy

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import json

def load_json_dataset(file_path):
    # 하나의 JSON 파일을 열어 문서별로 text, summary 정보를 추출
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    examples = []
    for doc in data["documents"]:
        sentences = []
        # "text"는 중첩 리스트 형태이므로 내부의 모든 sentence를 추출
        for sublist in doc["text"]:
            for item in sublist:
                sentences.append(item.get("sentence", ""))
        full_text = " ".join(sentences)
        # abstractive 요약은 첫번째 항목 사용 (없으면 빈 문자열)
        summary = doc["abstractive"][0] if doc["abstractive"] else ""
        examples.append({
            "text": full_text,
            "summary": summary,
        })
    return examples

In [ ]:
# 기본 경로 설정
base_path = "/content/drive/MyDrive/미션 스프린트/미션 12"

# law 관련 파일 하나만 불러오기 (코드상 변수명은 editorial이지만 주석은 law로 되어있음)
train_file = "train_original_editorial.json"
valid_file = "valid_original_editorial.json"

train_examples = load_json_dataset(os.path.join(base_path, train_file))
valid_examples = load_json_dataset(os.path.join(base_path, valid_file))

print("Train examples:", len(train_examples))
print("Validation examples:", len(valid_examples))

Train examples: 56760
Validation examples: 7008


In [ ]:
# train_examples 리스트의 첫 번째 항목(인덱스 0)을 확인합니다.
sample_data = train_examples[0]

print("=== [데이터 구조 확인] ===")
print("Keys:", sample_data.keys()) # 데이터가 어떤 키(Key)를 가지고 있는지 확인 ('text', 'summary')
print("-" * 30)

print("1. 원문 (Text):")
print(sample_data["text"][:300] + "...") # 내용이 기니까 앞부분 300자만 출력해서 봅니다.
print("-" * 30)

print("2. 요약 (Summary):")
print(sample_data["summary"])
print("-" * 30)

=== [데이터 구조 확인] ===
Keys: dict_keys(['text', 'summary'])
------------------------------
1. 원문 (Text):
이명박 대통령이 어제 30대 그룹 총수를 모아놓고 "시대적 요구는 역시 총수가 앞장서야 한다. 이미 상당한 변화의 조짐이 있다는 것을 고맙게 생각한다. 총수들께서 직접 관심을 가져주시면 빨리 전파돼 긍정적인 평가를 받을 수 있다고 본다"고 말했다. 언뜻 보아 무슨 말인지 불분명하나 이 대통령이 지난 8ㆍ15 연설 후 정몽준 의원, 정몽구 현대차 회장이 각각 2000억원과 5000억원을 기부한 사실과 '공생발전'이란 화두를 연결하면 금방 짐작이 간다. 다른 그룹 총수들도 좀 나서라고 은근히 떠민 것이다. 이 대통령은 기부에 대한 후속...
------------------------------
2. 요약 (Summary):
이명박 대통령은 어제 30대 그룹 총수를 모아놓고 시대적 요구는 역시 총수가 앞장서야 한다고 발언하며 기부문화 확산을 은근히 강조했으나 대통령의 강요나 포퓰리즘에 의한 압박보다는 자발적 문화로 구축해야 효과가 더 큰 법으로 방안 마련을 맡기고 시간을 줘야 할 것으로 보인다.
------------------------------


In [ ]:
# valid_examples 리스트의 첫 번째 항목을 확인합니다.
valid_sample = valid_examples[0]

print("=== [Valid 데이터 구조 확인] ===")
print("Keys:", valid_sample.keys()) # 여기도 'text', 'summary'가 나와야 합니다.
print("-" * 30)

print("1. Valid 원문 (Text):")
print(valid_sample["text"][:300] + "...") # 앞부분 300자만 출력
print("-" * 30)

print("2. Valid 요약 (Summary):")
print(valid_sample["summary"])
print("-" * 30)

# (참고) Train 데이터와 내용이 다른지 비교해보기
print("Train 데이터와 내용이 같은가요?", train_examples[0]['text'] == valid_examples[0]['text'])

=== [Valid 데이터 구조 확인] ===
Keys: dict_keys(['text', 'summary'])
------------------------------
1. Valid 원문 (Text):
더불어민주당 이해찬 대표가 30 일 오후 국회에서 기자간담회를 열고 조국 전 법무부 장관 사태와 관련해 "국민 여러분께 매우 송구하다"고 밝혔다. 더불어민주당 이해찬 대표가 30 일 기자간담회를 열고 '조국 사태'와 관련, "국민 여러분께 매우 송구하다"는 입장을 밝혔다. 이 대표는 "검찰 개혁이란 대의에 집중하다 보니, 국민 특히 청년이 느꼈을 불공정에 대한 상대적 박탈감, 좌절감을 깊이 있게 헤아리지 못했다"며 "여당 대표로서 무거운 책임감을 느낀다"고 머리를 숙였다. 조국 전 법무부 장관이 14 일 사퇴한 이후 이 대표가 당 ...
------------------------------
2. Valid 요약 (Summary):
이해찬 대표가 조국 사태와 관련 송구한 입장 표명이 과감한 인적 쇄신으로 이어져야 한다.
------------------------------
Train 데이터와 내용이 같은가요? False


# 2. 데이터셋 준비 및 Hugging Face Dataset 변환


## A. Dataset과 DatasetDict 만들기

In [ ]:
from datasets import Dataset, DatasetDict

# 1. 파이썬 리스트(train_examples)를 Hugging Face Dataset 객체로 변환합니다.
train_dataset = Dataset.from_list(train_examples)
valid_dataset = Dataset.from_list(valid_examples)

# 2. 훈련용과 검증용 데이터를 하나의 DatasetDict로 묶어서 관리합니다.
dataset = DatasetDict({
    "train": train_dataset,
    "validation": valid_dataset
})

# 변환된 데이터셋의 구조를 확인합니다.
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'summary'],
        num_rows: 56760
    })
    validation: Dataset({
        features: ['text', 'summary'],
        num_rows: 7008
    })
})


# 3. 토크나이저(Tokenizer) 설정 및 데이터 전처리

## 1) 토크나이저(Tokenizer) 불러오기 (Text to Number)
### 사전 학습된 KoBART 모델과 짝이 맞는 '번역기(Tokenizer)'를 가져옵니다.
 - 한국어 요약에 특화된 KoBART를 사용
 - BartTokenizerFast: 속도가 빠른(Fast) 버전의 토크나이저를 로드합니다.

In [ ]:
from transformers import BartTokenizerFast

# 1. 사용할 모델의 이름(Checkpoint)을 지정합니다.
checkpoint = "gogamza/kobart-base-v1"

# 2. 해당 모델에 맞는 Fast Tokenizer를 불러옵니다.
tokenizer = BartTokenizerFast.from_pretrained(checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'PreTrainedTokenizerFast'. 
The class this function is called from is 'BartTokenizerFast'.


## 2) 전처리 함수 정의
### B. 전처리 함수(Tokenize Function) 정의

In [ ]:
def tokenize_function(example):
    # 1. 원문(text)을 토큰화합니다. (입력 데이터)
    model_inputs = tokenizer(
        example["text"],
        max_length=512,  # 최대 길이 제한
        truncation=True  # 512보다 길면 뒷부분 자르기
    )

    # 2. 요약문(summary)을 토큰화합니다. (정답 데이터)
    labels = tokenizer(
        text_target=example["summary"],
        max_length=256,  # 요약문은 원문보다 짧게 설정
        truncation=True
    )

    # 3. 모델이 정답을 알 수 있도록 'labels' 키에 요약문의 숫자 코드를 넣습니다.
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

3) 데이터셋에 일괄 적용

In [ ]:
# 전체 데이터셋(dataset)에 tokenize_function을 적용합니다.
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/56760 [00:00<?, ? examples/s]

Map:   0%|          | 0/7008 [00:00<?, ? examples/s]

변환된 데이터셋 구조 확인

In [ ]:
# 1. 전체 데이터셋 구조 확인
print("=== [변환된 데이터셋 구조] ===")
print(tokenized_datasets)
print("-" * 50)

# 2. 실제 데이터 하나를 꺼내서 확인 (숫자로 변한 모습)
sample = tokenized_datasets["train"][0]
print("=== [첫 번째 데이터 샘플] ===")
print("Keys:", sample.keys()) # 기존 text, summary 외에 input_ids, attention_mask, labels가 추가됨
print("\nInput IDs (일부):", sample["input_ids"][:20]) # 숫자로 변환된 원문
print("Labels (일부):", sample["labels"][:20])       # 숫자로 변환된 요약문

=== [변환된 데이터셋 구조] ===
DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 56760
    })
    validation: Dataset({
        features: ['text', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 7008
    })
})
--------------------------------------------------
=== [첫 번째 데이터 샘플] ===
Keys: dict_keys(['text', 'summary', 'input_ids', 'attention_mask', 'labels'])

Input IDs (일부): [16954, 15690, 18899, 22309, 16058, 14281, 14925, 17453, 17396, 14050, 17486, 12124, 14730, 9698, 14833, 14281, 15026, 20163, 23190, 19553]
Labels (일부): [16954, 15606, 18899, 22309, 16058, 14281, 14925, 17453, 17396, 16687, 12124, 14730, 9698, 14833, 14281, 15026, 20163, 23190, 16141, 15778]


# 4. 모델 학습(Fine-tuning) 설정 및 실행

1) 학습 보조 도구 및 모델 준비

- A. 데이터 콜레이터 (Data Collator) 설정

    - 요약: 서로 다른 길이의 데이터를 배치(Batch) 단위로 묶을 때, 가장 긴 문장에 맞춰 길이를 똑같이 맞춰주는 '정렬 도구'입니다.

In [ ]:
from transformers import DataCollatorForSeq2Seq

# 학습 시 데이터를 미니 배치 단위로 묶고 패딩(padding)을 처리하는 도구입니다.
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer)

- B. 모델 로드 (BartForConditionalGeneration)
    - 요약: 요약 작업(조건부 생성)에 특화된 구조를 가진 KoBART 모델을 불러옵니다.

In [ ]:
from transformers import BartForConditionalGeneration

# 사전 학습된 KoBART 모델을 불러옵니다.
# "ConditionalGeneration"은 입력(원문)을 주면 출력(요약)을 생성하는 모델 구조를 말합니다.
model = BartForConditionalGeneration.from_pretrained(checkpoint)

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


model.safetensors:   0%|          | 0.00/495M [00:00<?, ?B/s]

- C. 학습 설정 (TrainingArguments) 정의
    - 요약: 모델을 어디에 저장할지, 성적표(Log)는 어떻게 출력할지 등 '학습 규칙'을 정합니다.

In [ ]:
from transformers import TrainingArguments, Trainer

# 학습과 관련된 설정을 정의합니다.
training_args = TrainingArguments(
    output_dir="test-trainer",  # 학습된 모델과 체크포인트가 저장될 폴더 이름
    report_to="none"            # 완드비(WandB) 같은 외부 로깅 툴을 끕니다 (실습용)
    per_device_train_batch_size=8,  # <--- 👈 기본값
    learning_rate=5e-5,             # (참고) 학습률도 안 적으면 기본값 0.00005
    num_train_epochs=3.0,           # (참고) 에폭도 안 적으면 기본값 3
)

In [ ]:
# Hugging Face의 Trainer 객체를 생성합니다.
trainer = Trainer(
    model=model,                         # 학습시킬 모델
    args=training_args,                  # 학습 설정
    train_dataset=tokenized_datasets["train"],      # 학습용 데이터
    eval_dataset=tokenized_datasets["validation"],  # 검증용 데이터
    data_collator=data_collator,         # 데이터 정렬 도구
    tokenizer=tokenizer,                 # 토크나이저
)

# 학습을 시작합니다!
trainer.train()

/tmp/ipython-input-148460996.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,2.477500
1000,2.292800
1500,2.251800
2000,2.206300
2500,2.199700
3000,2.173500
3500,2.157800
4000,2.148800
4500,2.146600
5000,2.111300


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'forced_eos_token_id': 1}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=21285, training_loss=1.8294067425825296, metrics={'train_runtime': 5643.1913, 'train_samples_per_second': 30.174, 'train_steps_per_second': 3.772, 'total_flos': 5.18182861787136e+16, 'train_loss': 1.8294067425825296, 'epoch': 3.0})

# 5. 모델 추론(Inference) 및 결과 확인

- A. 파이프라인(Pipeline)이란?
    - 요약: 복잡한 과정(전처리 -> 모델 입력 -> 결과 해석)을 버튼 하나로 압축해 주는 '자동화 도구'입니다.

In [ ]:
from transformers import pipeline

# 요약(summarization) 작업을 수행하는 파이프라인을 만듭니다.
# 방금 학습시킨 model과 tokenizer를 넣어줍니다.
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer)

Device set to use cuda:0


- B. 반복문을 통한 결과 확인
    - 요약: 검증 데이터(Validation) 중 10개를 뽑아서 원문과 모델이 만든 요약문을 비교해 봅니다.

- 앞선 결과에서 요약요약요약요약과 같은 것들이 반복되어나타났다. 이는 반복되는 것에 대한 벌점이 없고, 최소길이를 채우기 위해 쓸 데 없는 말을 넣었기 때문이다.
    - 이에 no_repeat_ngram_size, repetition_penalty, do_sample 등이 필요하다.

In [ ]:
# 검증 데이터셋의 첫 10개 예시에 대해 요약 결과를 출력합니다.
for i in range(10):
    # 1. 검증 데이터에서 원문(text)을 가져옵니다.
    sample_text = tokenized_datasets["validation"][i]["text"]

    # 2. 보기 좋게 출력하기 위한 프롬프트를 만듭니다.
    prompt = f"원문:\n{sample_text}\n요약:"

    print(f"예시 {i+1}:")
    print(prompt)

    # 3. 모델에게 요약을 시킵니다! (여기가 핵심)
    result = summarizer(
        prompt,
        max_length=128,          # 요약문 최대 길이
        min_length=10,           # 최소 길이 (너무 길게 잡으면 억지로 늘립니다)
        no_repeat_ngram_size=3,  # [중요] 3단어 이상 겹치는 구절 반복 금지
        repetition_penalty=2.0,  # [중요] 반복하면 벌점 부여 (확률 깎음)
        do_sample=False   # 창의적으로 짓지 말고(False), 가장 확률 높은 정답을 찾아라(Greedy)
    )

    # 4. 결과를 출력합니다.
    print(result[0]['summary_text'])
    print("-" * 50) # 구분선

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


예시 1:
원문:
더불어민주당 이해찬 대표가 30 일 오후 국회에서 기자간담회를 열고 조국 전 법무부 장관 사태와 관련해 "국민 여러분께 매우 송구하다"고 밝혔다. 더불어민주당 이해찬 대표가 30 일 기자간담회를 열고 '조국 사태'와 관련, "국민 여러분께 매우 송구하다"는 입장을 밝혔다. 이 대표는 "검찰 개혁이란 대의에 집중하다 보니, 국민 특히 청년이 느꼈을 불공정에 대한 상대적 박탈감, 좌절감을 깊이 있게 헤아리지 못했다"며 "여당 대표로서 무거운 책임감을 느낀다"고 머리를 숙였다. 조국 전 법무부 장관이 14 일 사퇴한 이후 이 대표가 당 안팎의 쇄신 요구에 대해 입장을 표명한 것은 이번이 처음이다. 청와대와 여당은 '조국 정국'을 거치며 분출된 '공정'과 '정의'의 민심을 받들어 검찰 개혁에 매진하겠다면서도 두 달간 극심한 분열과 갈등을 초래한데 대해선 진지하게 성찰하는 모습을 보이지 않았다. 그나마 초선인 이철희 의원이 "당이 대통령 뒤에 비겁하게 숨어 있었다"고 비판했고, 표창원 의원은 "책임을 느끼는 분들이 각자 형태로 그 책임감을 행동으로 옮겨야 할 때"라고 지적했다. 뒤늦게나마 이 대표가 자성의 목소리를 내긴 했으나 당 안팎의 쇄신 요구에 어떻게 응할지 구체적 플랜을 제시하지 못해 여전히 안이하다는 지적도 나온다. 이 대표는 28 일 윤호중 사무총장을 단장으로 하는 총선기획단을 발족했고 조만간 인재영입위원회도 출범시킬 계획이라고 밝혔다. 이 대표는 "민주당의 가치를 공유하는 참신한 인물을 영입해 준비된 정책과 인물로 승부하겠다"고 다짐했다. 하지만 당 일각에선 "총선기획단장을 비롯한 당직 인선부터 쇄신 의지를 보여야 한다"는 비판의 목소리가 나온다. 무조건 물러나는 게 능사는 아니지만 국정 혼선을 초래한 데 대해 당 지도부가 겸허하게 책임지는 모습을 보이는 게 쇄신의 출발점이 돼야 한다는 지적도 있다. 선거는 대중의 이해와 요구를 잘 대표하는 정치인을 뽑는 행위다. 민생을 외면하며 낡은 이념과 진영 싸움에 매몰된 구시대 인물들을 과감히 물갈

Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


더불어민주당 이해찬 대표가 30 일 기자간담회를 열고 조국 전 법무부 장관 사태와 관련해 "국민 여러분께 매우 송구하다"는 입장을 밝혔지만 당 일각에선 "총선기획단장을 비롯한 당직 인선부터 쇄신 의지를 보여야 한다"는 비판의 목소리가 나오며 민생을 외면하며 낡은 이념과 진영 싸움에 매몰된 구시대 인물들을 과감히 물갈이하라는 게 국민의 요구다.    이 대표의 이날 유감 표명이 여권 전반의 대대적인 인적 쇄신으로 이어지길 기대한다.  


제약: 새정치 민주 연합 대표로서 무거운 책임감을 느낀다"고 머리를 조아렸다.   원문:
더불어민주당 이해찬 대표의 30일 기자간담회 입장표명이 여권 전체 대대적 인적 쇄신의 출발점이 돼야 한다는 지적도 있다.   4 차 산업혁명의 거센 파고를 헤쳐나갈 전문성을 갖춘 젊고 유능한 인재들을 널리 구해야 한다.   고.유능한 인재를 널리 구해내야 한다.  인사 개혁으로 이어져야 할것이다.   라고 하였다. 네.민주당은 민생을 돌보지 않는 구시대 정치인을 과감히 교체하라는 국민의 요구를 받아들여야 한다.  .   제 1야당 이해찬대표의 입장 표명은 여권 전체의 대대적인 인적 쇄신을 기대한다  
--------------------------------------------------
예시 2:
원문:
탈원전 정책에서 비롯된 한국전력의 경영 부담이 결국 전기료 인상을 초래하게 됐다. 김종갑 한전 사장은 언론 인터뷰에서 "정부 정책에 따라 도입된 각종 전기료 특례할인을 모두 폐지하고 전기요금 원가공개 방안을 정부와 협의하겠다"고 밝혔다. 특례할인은 여름철 누진제, 주택용 절전 및 전기차 충전 등의 명목으로 전기료를 깎아주는 제도로서, 작년 1 조 1434 억원에 달했다. 이 제도가 없어지면 부담은 고스란히 국민에게 돌아가게 된다. 전기료 인상은 예견됐던 것이나 마찬가지다. 문재인 정부 출범 이후 탈원전 정책이 가속화되면서 한전 경영이 악화일로를 달려왔기 때문이다. 원전 가동을 줄이고 발전 단가가 높은 민간 발전사의 전기 구매를 늘린 탓

Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

대구경북 통합신공항 선정 기준을 두고 대구시와 경북도가 최종 절충 방안을 제시한 가운데 군위군수에 대한 경찰의 압수수색과 피의자 신분 조사를 둘러싸고 이전 추진에 악재로 작용하지 않을까 우려하는 목소리도 많으므로 지역 민심과 여론을 한곳으로 모아 내우외환에 휘둘리지 않도록 해야 한다. 


국가 정책과 법적 절차에 의해 결정된 신공항 사업에 문제를 제기하는 것은 불순한 의도의 정치 공세"라고 반박했다. 
부산경남 정치권의 신공항 딴죽걸기는 어제오늘의 일이 아니다. 
아니다. 『김해신공항 무산시키고 가덕도 신공항을 추진하려는 부울경 정치권과 현 정권의 복심을 대변하는것은 뻔뻔히 보여집니다...그러나 그럴수록 지역 민심은 한곳에 모아야 할것이다..『전면 재검토해야 할것입니다..이런저러한 상황에 휘둘려서는 안됩니다..(제목:사카요약:구체요약) 부산경남 정치권이여.안사카요약을 잊어서는 안된다고 본다..정당들은 말합니다..입니다몹시 조심해야 합니다..민주주의 기본원칙:국가 정책과 법률이 바로 잡습니다..
--------------------------------------------------
예시 8:
원문:
이주노동자들에게 적용되는 '출국 후 퇴직금 수령제도'로 인해 이주노동자들이 퇴직금을 받는데 어려움을 겪고 있다. 불법체류를 막기 위해 이주노동자가 출국한 뒤 퇴직금을 받도록 한 '출국만기보험 제도'가 퇴직금 미지급자를 양산하고 있는 것이다. 출국만기보험이란 사업주가 퇴직금을 지급하기 위해 노동자를 피보험자로 해 가입하는 보험이다. 사업주가 매달 통상임금의 8.3%를 보험회사에 적립한 것으로, 노동자가 퇴직 후 출국할 때 공항에서 받을 수 있다. 2014 년 정부는 불법체류 방지를 위해 이주노동자 퇴직금을 '퇴사 후 14 일 이내'에서 '출국 후 14 일 이내' 지급하는 것으로 바꿨다. 하지만 상당수 이주노동자들이 퇴직금 받는 방법을 제대로 모르고 있다. 이주노동자가 퇴직금을 받으려면 구비서류가 많은데다 퇴직금을 수령받는 절차도 복잡하다. 최근 이주노동자를 대상으

Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


이주노동자들이 불법체류를 막기 위해 이주노동자가 출국한 뒤 퇴직금을 받도록 한 '출국만기보험 제도'로 인해 퇴직금 미지급자를 양산하고 있는 가운데 이주노동자 관련 단체에서는 정부가 법을 개정한 취지가 불법체류자를 줄이기 위한 것이었으나 불법 체류자 감소에 큰 효과를 내지 못하는 만큼 출국 후 퇴직금 수령제도를 철폐해야 한다고 주장하고 있다.    근로기준법은 사업주가 '퇴직' 후 14 일 이내에 퇴직금을 지불해야 한다고 규정하고 있다.  그런데도 이주노동자에게만 그 시점을 다르게 적용하는 것은 기본권 침해라는 지적도 나오고 있어 하루빨리 해결책을 마련해 열악한 작업장에서 일하는 이주 노동자가 더이상 피해를 보지 않도록 해야 한다.   덧붙여 사업주가 노동자에게 잔여퇴직금이 없다, 공항에서 준다고 거짓말을 하는 사례도 빈번히 발생하고 있어 이에 대한 정부의 적절한 조치가 필요하다.   .   정부는 조속히 해결책을 마련하여 이주노동자의 피해를 막아야 한다.  라고 전했다.  
.   제도에 대한 적절한 조치도 필요해 보인다.  법제화를 서둘러야 할것이다.   이러한 문제들을 해결하기 위해서는 이주노동자들의 권리가 보호되어야 할것이다   이   이슈가 된다.   라는 말이 나오고 있다. 
--------------------------------------------------
예시 9:
원문:
법무부가 제정한 훈령 '형사사건 공개 금지 등에 관한 규정'을 둘러싼 논란이 뜨겁다. 당장 국민의 알권리와 정면 충돌하고 있다는 지적이 비등하고 있다. 법무부의 훈령은 사건 관계인의 인권보호를 위해 검찰의 수사 상황을 전면 비공개하고, 오보 언론에 대한 처벌 수단을 마련하겠다는 게 그 요지다. 구체적으로는 검사와 수사관 등 관계자가 기자와 개별적으로 만날 수 없도록 했다. 또 검찰수사와 관련해 오보를 내면 해당 언론사는 브리핑은 물론 검찰청 출입 자체가 제한된다. 다만 공보 담당자와는 만날 수 있다. 한마디로 기자는 알려주는 내용만 받아쓰라는 것이다. 언론 자유의 침해가 

Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


법무부가 '형사사건 공개 금지 등에 관한 규정'을 제정하여 피의사실 공개로 피의자 인권이 침해당하는 폐해를 막기 위한 조치라는 게 법무부 설명이지만 기자의 브리핑 참석과 출입 제한은 받아들이기 어려운 사안으로 정부가 규정 시행을 밀어붙인다면 비판 언론에 재잘을 물리려는 의도로 밖에 볼 수 없다.    정부는 규정 시행에 앞서 오보의 정도가 지나쳐 해당 언론사를 제재할 일이 있더라도 이는 언론 종사사들 간에 자율적으로 정하면 된다.   .   사법부의 판단 기준도 명확하지 않다.   기자 브리핑 참석 및 출입을 차단하고 받아쓰기만 강요한다면 더 이상 언론은 그 기능을 유지하기 어렵다.  MB정권의 적폐가 되살아 날 수도 있다.   고위공직자비리수사처(공수처)설립 취지에 어긋난다.  제 2의 노무현 전 대통령 사건 등 과거 적폐를 없애기 위해 노력해야 할 것이다.   라고 말했다.   입법 취지와도 맞지 않는다..   기사 삭제 등 언론중재법에 의한 권리 구제가 가능하다.  
   이 훈령에는 오보를 판단하는 기준도 분명하지 않다   문제   법령 시행을 밀어 붙이는것은 옳지 않
--------------------------------------------------
예시 10:
원문:
딱 한달이다. 두달도 이어가지 못했다. 5 개월만에 생산·소비·투자 등 경제 활동 3 대 지표가 모두 상승하며 희망을 줬던 산업활동동향(8 월)이 다시 한숨 속 위기감으로 빠져드는데는 한달이면 충분했다. 통계청이 31 일 발표한 '9 월 산업활동 동향'에서 지난달 전(全)산업 생산지수(계절조정계열)는 108.0 으로, 전월보다 0.4% 감소했다. 광공업 생산은 증가했지만, 도소매와 금융·보험업을 중심으로 서비스업 생산이 줄어든 것이 영향을 미쳤다. 소매판매는 음식료품과 의복 판매가 감소하면서 전월보다 2.2%줄어들었다. 감소폭은 2017 년 12 월(-2.4%) 이후 가장 컸다. 이제는 익숙해져 그러려니 하지만 통계청의 설명은 여지없이 기저효과와 날씨 때문이란 설명으로 일관한다.

ROUGE 점수 평가 코드

In [ ]:
import torch
import gc

# 1. 꽉 찬 GPU 메모리를 비워줍니다. (쓰레기 수집)
gc.collect()
torch.cuda.empty_cache()

# 2. 평가(Evaluation) 설정을 변경합니다.
# 배치 사이즈를 줄여서 GPU 부담을 낮춥니다. (기존 8 -> 4 또는 2로 감소)
trainer.args.per_device_eval_batch_size = 4

# [핵심] GPU 메모리가 터지는 것을 막기 위해, 계산된 결과는 즉시 CPU로 옮깁니다.
# 이 설정이 없으면 모든 결과가 끝날 때까지 GPU에 쌓여서 OOM이 발생합니다.
trainer.args.eval_accumulation_steps = 1

print("메모리 청소 완료 & 평가 설정 변경됨!")

메모리 청소 완료 & 평가 설정 변경됨!


In [ ]:
# 1. 평가에 필요한 라이브러리 설치
!pip install evaluate rouge_score

import evaluate
import numpy as np

# ROUGE 메트릭 불러오기
rouge = evaluate.load('rouge')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ROUGE 점수 계산
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    # 보기 좋게 소수점 4자리까지 반올림
    result = {k: round(v * 100, 4) for k, v in result.items()}
    return result

# Trainer에 평가 함수를 추가하여 검증 데이터셋에 대한 점수 확인
# (이미 학습된 trainer 객체를 활용)
trainer.compute_metrics = compute_metrics
metrics = trainer.evaluate()

print("=== 모델 최종 성능 평가 (ROUGE Score) ===")
print(metrics)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


NameError: name 'trainer' is not defined